# Ti-6Al-4V Cutting Thermal Model

Flash temperature, surface and subsurface temperature, and thermally induced residual stress for orthogonal cutting of Ti-6Al-4V. This notebook is self-contained: every function is defined in a code cell below, and all derivations are in the text cells. Only `numpy`, `scipy` and `matplotlib` are required.

**Pipeline**

1. Cutting force $F_c$ from a Kienzle fit
2. Flash temperature $\Delta T_{flash}$ and Peclet number $Pe$ (fixed-point iteration)
3. Normalized surface temperature profile
4. Subsurface 2D temperature field
5. Critical temperature and residual stress
6. Machining sweeps (force, temperature and critical speed vs feed and speed)

Sources: Thermal Modeling Lecture Slides (rev4), `Subsurface_thermal.m`, the ME599 Ti-6Al-4V workbook, Cutting Force Data.xlsx, Liu et al. 2004 (ASME), Theraroz et al. 2025.

In [ ]:
%matplotlib inline
from dataclasses import dataclass
from functools import lru_cache

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy.special import erf, k0e, k1e

T0 = 20.0  # ambient temperature (deg C), fixed

## Inputs

In [ ]:
v_m_min = 60.0   # cutting speed (m/min)
h_mm    = 0.05   # feed / uncut chip thickness (mm)
b_um    = 200.0  # contact half-width b (um); contact length = 2b
w_mm    = 3.0    # width of cut (mm)

SWEEP_FEEDS = (0.01, 0.05, 0.10)  # feeds (mm) for the flash-temperature-vs-speed lines

## Material properties

Ti-6Al-4V properties are polynomial fits in temperature $T$ (°C), taken from the reference workbook.

| Property | Expression | Units |
|---|---|---|
| Density | $\rho = 4500$ | kg/m³ |
| Conductivity | $k(T) = 1.12\times10^{-5}\,T^2 + 0.00982\,T + 6.4$ | W/(m·K) |
| Specific heat | $c_p(T) = 1.71\times10^{-6}\,T^3 - 0.00173\,T^2 + 0.6\,T + 536$ | J/(kg·K) |
| Young's modulus | $E(T) = -40\,T + 100000$ | MPa |
| Thermal expansion | $\alpha_{th}(T) = \left(1.23\times10^{-6}\,T^2 + 2.87\times10^{-3}\,T + 4.57\right)\times 1.8\times10^{-6}$ | 1/°C |
| Yield strength | $\sigma_y(T) = \left(0.005752\,T^4 - 10.67\,T^3 + 5597\,T^2 - 1.79\times10^{6}\,T + 1.056\times10^{9}\right)\times10^{-6}$ | MPa |
| Poisson ratio | $\nu = 0.32$ | |
| Liquidus | $T_{melt} = 1660$ | °C |

The residual-stress fit above the critical temperature is $\sigma_{RS} = 2.8788\,T - 1365.2$ MPa (used in the residual stress section).

In [ ]:
TI64 = dict(
    rho=4500.0,
    k=lambda T: 1.12e-5 * T**2 + 0.00982 * T + 6.4,
    cp=lambda T: 1.71e-6 * T**3 - 0.00173 * T**2 + 0.6 * T + 536,
    E=lambda T: -40.0 * T + 100000.0,
    alpha=lambda T: (1.23e-6 * T**2 + 2.87e-3 * T + 4.57) * 1.8e-6,
    sigma_y=lambda T: (0.005752 * T**4 - 10.67 * T**3 + 5597.0 * T**2
                       - 1.79e6 * T + 1.056e9) * 1e-6,
    poisson=0.32,
    rs_fit=(2.8788, 1365.2),
    T_melt=1660.0,
)

## 1. Cutting force (Kienzle fit)

The tangential cutting force is the product of the specific cutting force $k_c$, the uncut chip thickness $h$ and the width of cut $w$:

$$F_c = k_c\,h\,w, \qquad k_c(h) = C\,h^{\,n}$$

with $C = 1792.69$ N/mm² and $n = -0.11672$ ($h$ in mm), fit to Cutting Force Data.xlsx. The negative exponent is the size effect: thinner chips need more energy per unit chip area. $F_c$ is the heat input to the thermal model.

In [ ]:
KIENZLE_TI64 = (1792.69, -0.11672)


def kienzle_force(h_mm, w_mm):
    C, n = KIENZLE_TI64
    return C * h_mm**n * h_mm * w_mm


Fc = float(kienzle_force(h_mm, w_mm))
print(f"Kienzle C = {KIENZLE_TI64[0]}, n = {KIENZLE_TI64[1]}")
print(f"Fc = {Fc:.1f} N")

## 2. Flash temperature

Heat enters through a moving band source of half-width $b$ sliding at speed $v$. The Peclet number compares advection to conduction:

$$Pe = \frac{v\,\rho\,b\,c_p}{2k}$$

Low $Pe$: heat conducts away faster than the source moves. High $Pe$: the source outruns the heat and heating is confined to a thin surface layer.

The peak temperature rise has a separate correlation on each side of $Pe = 5$:

- $Pe < 5$ (fit to the exact moving-source integral):

$$\Delta T_{flash} = a_{low}\,(0.159)\,C_4\,\frac{2F_c}{\rho\,c_p\,w\,b}, \qquad C_4 = 0.00527\,Pe^3 - 0.192\,Pe^2 + 2.39\,Pe$$

- $Pe \ge 5$ (fast-source asymptote, $\Delta T \propto 1/\sqrt{Pe}$):

$$\Delta T_{flash} = a_{high}\,(0.399)\,\frac{2F_c\,v}{k\,w}\sqrt{\frac{k}{\rho\,c_p\,v\,b}}$$

The factor $2F_c v$ is the heat generation rate. $a_{low}$ and $a_{high}$ are calibration multipliers (both 1 by default). The Excel workbooks omit the factor 2 on the high-$Pe$ branch while `Subsurface_thermal.m` keeps it; this model follows the MATLAB script and uses it on both branches.

**Temperature-dependent properties.** $k$ and $c_p$ depend on temperature, but $\Delta T_{flash}$ is the unknown. This is solved by damped fixed-point iteration:

1. Start from $T_{guess} = (T_0 + T_{melt})/2$.
2. Evaluate $k(T_{guess})$, $c_p(T_{guess})$, then $Pe$ and $\Delta T_{flash}$.
3. Update $T_{guess} \leftarrow \left(\Delta T_{flash} + T_{guess}\right)/2$. The averaging prevents oscillation.
4. Stop when $|\Delta T_{flash} - T_{guess}| < 10^{-3}$ (capped at 200 iterations).

The peak surface temperature is $T_0 + \Delta T_{flash}$.

In [ ]:
CALIBRATION = dict(a_low=1.0, a_high=1.0)


def peclet_number(v_m_s, rho, b_m, cp, k):
    return v_m_s * rho * b_m * cp / (2.0 * k)


@dataclass(frozen=True)
class FlashResult:
    T_flash: float
    Pe: float
    iterations: int
    converged: bool
    k: float
    cp: float
    history: tuple = ()


def flash_temperature(v_m_s, Fc_N, w_m, b_m, T_ambient=20.0, T_initial=None,
                      max_iter=200, tol=1e-3, a_low=None, a_high=None):
    a_low = CALIBRATION["a_low"] if a_low is None else a_low
    a_high = CALIBRATION["a_high"] if a_high is None else a_high
    m = TI64
    rho = m["rho"]
    T_guess = (T_ambient + m["T_melt"]) / 2.0 if T_initial is None else T_initial
    converged = False
    Pe, T_flash = float("nan"), T_guess
    k, cp = m["k"](T_guess), m["cp"](T_guess)
    iteration = 0
    history = [float(T_guess)]

    for iteration in range(1, max_iter + 1):
        k, cp = m["k"](T_guess), m["cp"](T_guess)
        Pe = peclet_number(v_m_s, rho, b_m, cp, k)
        if Pe < 5:
            C4 = 0.00527 * Pe**3 - 0.192 * Pe**2 + 2.39 * Pe
            T_flash = a_low * 0.159 * C4 * (2.0 * Fc_N) / (rho * cp * w_m * b_m)
        else:
            T_flash = (a_high * 0.399 * (2.0 * Fc_N * v_m_s) / (k * w_m)
                       * (k / (rho * cp * v_m_s * b_m)) ** 0.5)
        history.append(float(T_flash))
        if abs(T_flash - T_guess) < tol:
            converged = True
            break
        T_guess = (T_flash + T_guess) / 2.0

    return FlashResult(T_flash, Pe, iteration, converged, k, cp, tuple(history))

## 3. Surface temperature profile (1D)

The surface temperature along the cut follows the moving band-source solution (Liu 2004, Eq. 2.13). With $\xi = x/b$ and

$$P_+ = \frac{Pe\,(\xi + 1)}{2}, \qquad P_- = \frac{Pe\,(\xi - 1)}{2}$$

the unnormalized shape $f(\xi)$ has three branches ($K_0$, $K_1$ are modified Bessel functions of the second kind):

$$f(\xi) = \begin{cases}
(\xi+1)\,e^{P_+}\left[K_0(-P_+) - K_1(-P_+)\right] + (1-\xi)\,e^{P_-}\left[K_0(-P_-) - K_1(-P_-)\right], & \xi < -1 \\[4pt]
(\xi+1)\,e^{P_+}\left[K_0(P_+) + K_1(P_+)\right] + (1-\xi)\,e^{P_-}\left[K_0(-P_-) - K_1(-P_-)\right], & -1 \le \xi \le 1 \\[4pt]
(\xi+1)\,e^{P_+}\left[K_0(P_+) + K_1(P_+)\right] + (1-\xi)\,e^{P_-}\left[K_0(P_-) + K_1(P_-)\right], & \xi > 1
\end{cases}$$

The shape is divided by its own maximum so the peak equals 1, then scaled by the flash temperature:

$$T(x) = T_0 + \Delta T_{flash}\,\frac{f(\xi)}{f_{max}}$$

**Numerical notes.** At large $Pe$ the factor $e^{P}$ overflows, so the code uses the scaled Bessel functions $\mathrm{k0e}(P) = e^{|P|}K_0(P)$ and $\mathrm{k1e}(P) = e^{|P|}K_1(P)$. For $P>0$, $e^{P}K(P)$ is computed directly as $\mathrm{k0e}+\mathrm{k1e}$. For $P<0$, $e^{P}\left[K_0(-P) - K_1(-P)\right] = e^{-2|P|}\left[\mathrm{k0e}(|P|) - \mathrm{k1e}(|P|)\right]$. The $K$ functions are singular at 0, so $P$ is nudged to $\pm10^{-9}$ when it is exactly zero. `Subsurface_thermal.m` line 52 has a dead middle branch (a chained comparison that is never true); the three-branch form above is the correct one.

In [ ]:
def _safe(z):
    return np.where(np.abs(z) < 1e-9, np.where(z >= 0, 1e-9, -1e-9), z)


def normalized_shape(x_over_b, Pe):
    x = np.asarray(x_over_b, dtype=float)
    Pp = _safe(Pe * (x + 1.0) / 2.0)
    Pm = _safe(Pe * (x - 1.0) / 2.0)

    def pos(P):
        P = np.abs(P)
        return k0e(P) + k1e(P)

    def neg(P):
        P = np.abs(P)
        return np.exp(-2.0 * P) * (k0e(P) - k1e(P))

    plus_neg, plus_pos = (x + 1.0) * neg(Pp), (x + 1.0) * pos(Pp)
    minus_neg, minus_pos = (1.0 - x) * neg(Pm), (1.0 - x) * pos(Pm)

    raw = np.select([x < -1.0, x > 1.0],
                    [plus_neg + minus_neg, plus_pos + minus_pos],
                    default=plus_pos + minus_neg)
    return raw / np.max(raw)


DEFAULT_X_OVER_B = np.linspace(-3.0, 5.0, 2000)
DEFAULT_Z_OVER_B = np.linspace(0.0, 4.0, 200)


def _flash(v_m_min, Fc_N, w_mm, b_um, T_ambient):
    return flash_temperature(v_m_min / 60.0, Fc_N, w_mm / 1000.0,
                             b_um * 1e-6, T_ambient)


@dataclass(frozen=True)
class ProfileResult:
    x_over_b: np.ndarray
    shape: np.ndarray
    T_actual_C: np.ndarray
    Pe: float
    T_flash: float
    peak_x_over_b: float
    converged: bool
    iterations: int
    k: float
    cp: float
    history: tuple


def solve(v_m_min, Fc_N, w_mm, b_um, T_ambient=20.0, x_over_b=None):
    x = DEFAULT_X_OVER_B if x_over_b is None else np.asarray(x_over_b, dtype=float)
    f = _flash(v_m_min, Fc_N, w_mm, b_um, T_ambient)
    shape = normalized_shape(x, f.Pe)
    return ProfileResult(x, shape, T_ambient + f.T_flash * shape, f.Pe, f.T_flash,
                         float(x[np.argmax(shape)]), f.converged, f.iterations,
                         f.k, f.cp, f.history)

In [ ]:
r = solve(v_m_min, Fc, w_mm, b_um, T0)
print(f"Peak surface T       = {T0 + r.T_flash:.1f} C")
print(f"Flash rise dT        = {r.T_flash:.1f} K")
print(f"Peclet number        = {r.Pe:.3f}")
print(f"k, cp at convergence = {r.k:.2f} W/(m K), {r.cp:.1f} J/(kg K)")
print(f"Density              = {TI64['rho']:g} kg/m^3")
print(f"Initial guess        = {r.history[0]:.0f} C   (T_melt = {TI64['T_melt']:g} C)")
print(f"Iterations           = {r.iterations}  (converged: {r.converged})")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(len(r.history)), r.history, "o-")
ax.axhline(r.T_flash, color="gray", ls="--", lw=1)
ax.set(title="Flash temperature convergence", xlabel="Iteration (0 = initial guess)", ylabel="ΔT_flash (K)")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(alpha=0.3)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
a1.plot(r.x_over_b, r.shape)
a1.plot(r.peak_x_over_b, 1.0, "o", color="C0")
a1.set(title=f"Normalized surface shape (Pe = {r.Pe:.2f})", xlabel="x / b", ylabel="(T − T₀) / ΔT_flash", ylim=(0, 1.1))
a2.plot(r.x_over_b, r.T_actual_C, color="C1")
a2.plot(r.peak_x_over_b, T0 + r.T_flash, "o", color="C1")
a2.set(title="Actual surface temperature", xlabel="x / b", ylabel="Temperature (°C)")
for a in (a1, a2):
    a.grid(alpha=0.3)
print(f"Peak at x/b = {r.peak_x_over_b:.2f}")

## 4. Subsurface temperature (2D)

The 2D field is approximated as separable: the surface profile times a 1D transient-conduction decay with depth $z$. A point on the surface sits under the source for the time it takes the source (width $2b$) to pass:

$$t = \frac{2b}{v}, \qquad \alpha = \frac{k}{\rho\,c_p}, \qquad \delta = \sqrt{4\alpha t}$$

where $\delta$ is the thermal diffusion length. Then

$$T(x, z) = \max\!\left(T_0,\; T_{surf}(x)\left[1 - \operatorname{erf}\!\left(\frac{z}{\delta}\right)\right]\right), \qquad T_{surf}(x) = T_0 + \Delta T_{flash}\,\frac{f(\xi)}{f_{max}}$$

As in the reference slides, MATLAB script and workbook, the attenuation multiplies the whole surface temperature (ambient included) and the result is floored at $T_0$. $k$ and $c_p$ are evaluated at the converged flash temperature.

In [ ]:
def subsurface_temperature(x_over_b, z_over_b, Pe, T_flash, T_ambient,
                           k, cp, rho, v_m_s, b_m):
    shape = normalized_shape(x_over_b, Pe)
    T_surface = T_ambient + T_flash * shape
    diffusion_length = np.sqrt(4.0 * (k / (rho * cp)) * (2.0 * b_m / v_m_s))
    atten = 1.0 - erf(np.asarray(z_over_b)[:, None] * b_m / diffusion_length)
    return np.maximum(T_ambient, atten * T_surface[None, :])

## 5. Critical temperature and residual stress

**Thermoelastic stress.** A surface element that is heated but constrained from expanding develops a biaxial compressive stress of magnitude

$$\sigma_{th}(T) = \frac{E(T)\,\alpha_{th}(T)\,(T - T_0)}{1 - \nu}$$

**Critical temperature.** $T_{crit}$ is the lowest temperature at which the thermoelastic stress first exceeds the temperature-dependent yield strength:

$$\sigma_{th}(T_{crit}) > \sigma_y(T_{crit})$$

The code scans $T$ from $T_0$ to 2000 °C in 20000 steps and takes the first point where the inequality holds (infinity if it never does). Above this temperature the material yields in compression while hot and ends up in tension after cooling.

**Residual stress.** Above $T_{crit}$ the workbook gives a linear fit; below it the residual stress is zero:

$$\sigma_{RS}(T) = \begin{cases} 2.8788\,T - 1365.2 \ \text{MPa}, & T > T_{crit} \\ 0, & T \le T_{crit} \end{cases}$$

The two lines are fitted independently and not forced to zero at $T_{crit}$, so small negative values just above $T_{crit}$ are a fit artifact.

`solve_2d` evaluates the temperature along the column at the flank / tool-exit point $x/b = 1$ and converts it to residual stress versus depth.

In [ ]:
def thermoelastic_stress(T, T_ambient=20.0):
    T = np.asarray(T, dtype=float)
    return TI64["E"](T) * TI64["alpha"](T) * (T - T_ambient) / (1.0 - TI64["poisson"])


@lru_cache(maxsize=32)
def critical_temperature(T_ambient=20.0, T_max=2000.0, n=20000):
    T = np.linspace(T_ambient, T_max, n)
    exceeds = thermoelastic_stress(T, T_ambient) > TI64["sigma_y"](T)
    return float(T[np.argmax(exceeds)]) if exceeds.any() else float("inf")


def residual_stress(T, T_critical):
    slope, intercept = TI64["rs_fit"]
    T = np.asarray(T, dtype=float)
    return np.where(T > T_critical, slope * T - intercept, 0.0)


@dataclass(frozen=True)
class TwoDResult:
    x_over_b: np.ndarray
    z_over_b: np.ndarray
    T_field_C: np.ndarray
    T_flank_profile_C: np.ndarray
    residual_stress_MPa: np.ndarray
    T_critical_C: float
    flank_x_over_b: float
    Pe: float
    T_flash: float
    converged: bool


def solve_2d(v_m_min, Fc_N, w_mm, b_um, T_ambient=20.0,
             x_over_b=None, z_over_b=None, flank_x_over_b=1.0):
    x = DEFAULT_X_OVER_B if x_over_b is None else np.asarray(x_over_b, dtype=float)
    z = DEFAULT_Z_OVER_B if z_over_b is None else np.asarray(z_over_b, dtype=float)
    f = _flash(v_m_min, Fc_N, w_mm, b_um, T_ambient)

    field = subsurface_temperature(x, z, f.Pe, f.T_flash, T_ambient, f.k, f.cp,
                                   TI64["rho"], v_m_min / 60.0, b_um * 1e-6)
    flank_idx = int(np.argmin(np.abs(x - flank_x_over_b)))
    T_flank = field[:, flank_idx]
    T_crit = critical_temperature(T_ambient)
    return TwoDResult(x, z, field, T_flank, residual_stress(T_flank, T_crit),
                      T_crit, float(x[flank_idx]), f.Pe, f.T_flash, f.converged)

In [ ]:
X = np.linspace(-3.0, 5.0, 801)   # contains x/b = 1 (flank) exactly
Z = 4.0 * np.linspace(0.0, 1.0, 200) ** 2   # fine near the surface, where RS lives
r2 = solve_2d(v_m_min, Fc, w_mm, b_um, T0, x_over_b=X, z_over_b=Z)
Tc = r2.T_critical_C
rs_field = residual_stress(r2.T_field_C, Tc)
yielded = r2.z_over_b[r2.residual_stress_MPa > 0]

print(f"Critical T (yield)  = {Tc:.1f} C")
print(f"Surface RS at flank = {r2.residual_stress_MPa[0]:.1f} MPa")
print(f"Yielded depth       = {yielded.max() * b_um if yielded.size else 0:.1f} um")

fig, ax = plt.subplots(figsize=(8, 4))
mesh = ax.pcolormesh(X, Z, r2.T_field_C, cmap="Oranges", shading="auto")
fig.colorbar(mesh, ax=ax, label="Temperature (°C)")
if rs_field.max() > 0:
    ax.contour(X, Z, r2.T_field_C, levels=[Tc], colors="k", linestyles="dashed")
    ax.plot([], [], "k--", label=f"T = T_crit ({Tc:.0f} °C)")
    for color, level in zip(["C0", "C2", "C4", "C9"], MaxNLocator(4).tick_values(0, rs_field.max())[1:-1]):
        ax.contour(X, Z, rs_field, levels=[level], colors=[color])
        ax.plot([], [], color=color, label=f"RS = {level:.0f} MPa")
ax.axvline(1.0, color="gray", ls=":")
ax.plot([], [], ":", color="gray", label="Flank x/b = 1")
ax.legend(loc="upper left", bbox_to_anchor=(1.3, 1.0), frameon=False)
ax.set(title="Subsurface temperature with residual stress", xlabel="x / b", ylabel="z / b (depth)", ylim=(0.9, 0))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(r2.z_over_b, r2.residual_stress_MPa, color="C1")
ax.set(title=f"Residual stress at flank (x/b = {r2.flank_x_over_b:g})", xlabel="z / b (depth)",
       ylabel="Residual stress (MPa)", xlim=(0, 1), ylim=(0, None))
ax.grid(alpha=0.3)

## 6. Machining sweeps

Force comes from the Kienzle fit; contact half-width $b$ and width of cut $w$ are held fixed. Speeds sweep 0.1-5 m/s (6-300 m/min). The black points are measured peak temperatures at $h = 0.05$ mm from the reference workbook.

**Critical speed.** $v_{crit}(h)$ is the lowest cutting speed at which the flank surface temperature at $x/b = 1$ exceeds $T_{crit}$, i.e. the speed above which thermal yielding leaves tensile surface residual stress:

$$T_0 + \Delta T_{flash}(v, h)\,\frac{f(1)}{f_{max}} > T_{crit}$$

It is found by a log-spaced scan of 60 speeds between 0.01 and 10 m/s, then 40 geometric bisection steps, $v_{mid} = \sqrt{v_{lo}\,v_{hi}}$, on the bracketing interval. The result is `nan` if the threshold is never reached by 10 m/s, and 0.01 m/s if it is already exceeded there.

**Power-law fit.** The critical speeds are fitted as

$$v_{crit} = a\,h^{\,n} \quad\Longleftrightarrow\quad \ln v_{crit} = \ln a + n\,\ln h$$

by linear least squares in log-log space over the finite points.

In [ ]:
_FLANK_X = np.linspace(-3.0, 5.0, 801)  # contains x/b = 1.0 exactly
_FLANK_IDX = 500


def force_vs_feed(h_mm, w_mm=3.0):
    return kienzle_force(np.asarray(h_mm, dtype=float), w_mm)


def _flash_sweep(v_m_s, h_mm, w_mm, b_um, T_ambient):
    Fc = float(kienzle_force(h_mm, w_mm))
    return [flash_temperature(v, Fc, w_mm / 1000.0, b_um * 1e-6, T_ambient)
            for v in np.atleast_1d(v_m_s)]


def flash_vs_speed(v_m_s, h_mm, w_mm=3.0, b_um=200.0, T_ambient=20.0):
    return np.array([T_ambient + f.T_flash
                     for f in _flash_sweep(v_m_s, h_mm, w_mm, b_um, T_ambient)])


def pe_vs_speed(v_m_s, h_mm, w_mm=3.0, b_um=200.0, T_ambient=20.0):
    return np.array([f.Pe for f in _flash_sweep(v_m_s, h_mm, w_mm, b_um, T_ambient)])


def _flank_surface_temperature(v_m_s, h_mm, w_mm, b_um, T_ambient):
    Fc = float(kienzle_force(h_mm, w_mm))
    f = flash_temperature(v_m_s, Fc, w_mm / 1000.0, b_um * 1e-6, T_ambient)
    return T_ambient + f.T_flash * normalized_shape(_FLANK_X, f.Pe)[_FLANK_IDX]


def critical_speed(h_mm, w_mm=3.0, b_um=200.0, T_ambient=20.0,
                   v_min=0.01, v_max=10.0):
    Tc = critical_temperature(T_ambient)
    over = lambda v: _flank_surface_temperature(v, h_mm, w_mm, b_um, T_ambient) > Tc
    grid = np.geomspace(v_min, v_max, 60)
    hits = [i for i, v in enumerate(grid) if over(v)]
    if not hits:
        return float("nan")
    if hits[0] == 0:
        return float(v_min)
    lo, hi = grid[hits[0] - 1], grid[hits[0]]
    for _ in range(40):
        mid = np.sqrt(lo * hi)
        lo, hi = (lo, mid) if over(mid) else (mid, hi)
    return float(hi)


def critical_speed_fit(h_mm, v_crit):
    h, v = np.asarray(h_mm, dtype=float), np.asarray(v_crit, dtype=float)
    ok = np.isfinite(v)
    n, ln_a = np.polyfit(np.log(h[ok]), np.log(v[ok]), 1)
    return float(np.exp(ln_a)), float(n)

In [ ]:
h = np.linspace(0.01, 0.12, 56)
v = 0.1 * np.arange(1, 51)                  # m/s
hc = np.linspace(0.01, 0.12, 23)
vc = np.array([critical_speed(x, w_mm, b_um, T0) for x in hc]) * 60.0   # m/min
ok = np.isfinite(vc)
a, n = critical_speed_fit(hc, vc)
v_crit = critical_speed(h_mm, w_mm, b_um, T0) * 60.0

print(f"Critical speed at h = {h_mm} mm: " + (f"{v_crit:.1f} m/min" if np.isfinite(v_crit) else "> 600 m/min"))
print(f"Critical speed fit: v = {a:.4g} * h^{n:.3f}  (v in m/min, h in mm)")

data_v = (20, 40, 60, 80, 100)              # m/min
data_T = (360, 400, 430, 470, 490)          # deg C, h = 0.05 mm

fig, axs = plt.subplots(1, 4, figsize=(21, 4))
axs[0].plot(h, force_vs_feed(h, w_mm))
axs[0].set(title=f"Cutting force vs feed (w = {w_mm:g} mm)", xlabel="Feed h (mm)", ylabel="Cutting force Fc (N)")
for hf in SWEEP_FEEDS:
    axs[1].plot(v, pe_vs_speed(v, hf, w_mm, b_um, T0), label=f"h = {hf:g} mm")
    axs[2].plot(v * 60, flash_vs_speed(v, hf, w_mm, b_um, T0), label=f"h = {hf:g} mm")
axs[1].set(title="Peclet number vs cutting speed", xlabel="Cutting speed (m/s)", ylabel="Peclet number", xlim=(0, 5), ylim=(0, None))
axs[2].plot(data_v, data_T, "ko", markersize=5, label="Data (h = 0.05 mm)")
axs[2].set(title="Peak temperature vs cutting speed", xlabel="Cutting speed (m/min)", ylabel="Peak temperature (°C)", xlim=(0, 300), ylim=(0, None))
axs[3].plot(hc[ok], a * hc[ok] ** n, "--", color="gray", label=f"v = {a:.3g}·h^{n:.3f}")
axs[3].plot(hc[ok], vc[ok], "o", color="C1", label="model")
axs[3].set(title="Critical speed for tensile surface RS", xlabel="Feed h (mm)", ylabel="Critical speed (m/min)", xlim=(0, 0.125), ylim=(0, None))
for ax in axs:
    ax.grid(alpha=0.3)
    if ax is not axs[0]:
        ax.legend(frameon=False, fontsize=8)